# TalentPulse — Feature Engineering & Ensemble Module (`feature_engineering.py`)

Notebook conversion of `../feature_engineering.py` (original `.py` file is kept intact so `joblib.load()` can unpickle saved models cleanly from `app.py` and scripts).

### Components Included
- **Constants:** `LEGACY_18_FEATURES`, `DROPPED_RAW_COLUMNS`, `TRAVEL_SCORE_MAP`
- **`HRFeatureEngineer`:** Leak-free HR domain feature transformer with dual-mode input (full raw DataFrame **or** legacy 18-column `app.py` array)
- **`build_preprocessor`:** `HRFeatureEngineer` -> `ColumnTransformer(PowerTransformer(yeo-johnson) + OneHotEncoder)`
- **`WeightedSoftVotingClassifier`:** Out-of-fold SLSQP simplex-optimized soft-voting ensemble classifier

## 1. Imports & Feature Schema Constants

In [ ]:
"""
TalentPulse Reusable Feature Engineering & Ensemble Components.

Kept in an importable top-level module (not __main__) so serialized joblib
artifacts (`models/scaler.pkl`, `models/voting_ensemble_model.pkl`,
`models/stacking_ensemble_model.pkl`, `models/best_pipeline.pkl`) can be
unpickled cleanly by `app.py`, inference services, or test scripts.
"""

from __future__ import annotations

from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer


# Legacy 18-feature schema used by app.py (order preserved exactly)
LEGACY_18_FEATURES: List[str] = [
    "Age",
    "DistanceFromHome",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "MonthlyIncome",
    "NumCompaniesWorked",
    "PercentSalaryHike",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager",
    "MaritalStatus_Single",
    "OverTime_Yes",
]

# Columns dropped during feature engineering (zero variance, pure ID, or pure noise rates)
DROPPED_RAW_COLUMNS: List[str] = [
    "EmployeeCount",
    "EmployeeNumber",
    "Over18",
    "StandardHours",
    "DailyRate",
    "HourlyRate",
    "MonthlyRate",
]

TRAVEL_SCORE_MAP: Dict[str, int] = {
    "Non-Travel": 0,
    "Travel_Rarely": 1,
    "Travel_Frequently": 2,
}

## 2. Leak-Free HR Domain Feature Engineer (`HRFeatureEngineer`)

In [ ]:
class HRFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Leak-free HR domain feature engineering transformer.

    Computes group-level compensation & tenure medians exclusively during `.fit()`
    on training folds to avoid any train/validation/test leakage.

    Also supports dual-mode input for backward compatibility:
      - Full raw DataFrame from `data/employee_attrition.csv` (recommended production mode)
      - Legacy 18-column numpy array / DataFrame matching `app.py` (graceful fallback
        that imputes omitted categorical/numeric fields using training-set modes/medians)
    """

    def __init__(self, drop_rate_columns: bool = True) -> None:
        self.drop_rate_columns = drop_rate_columns

    def fit(self, X: pd.DataFrame, y: Optional[Any] = None) -> "HRFeatureEngineer":
        df = self._ensure_dataframe(X, fitting=True)

        # Store training-set reference statistics for leak-free relative features
        self.role_income_med_: Dict[str, float] = (
            df.groupby("JobRole")["MonthlyIncome"].median().to_dict()
        )
        self.level_income_med_: Dict[int, float] = (
            df.groupby("JobLevel")["MonthlyIncome"].median().to_dict()
        )
        self.dept_income_med_: Dict[str, float] = (
            df.groupby("Department")["MonthlyIncome"].median().to_dict()
        )
        self.role_tenure_med_: Dict[str, float] = (
            df.groupby("JobRole")["YearsAtCompany"].median().to_dict()
        )
        self.global_income_med_: float = float(df["MonthlyIncome"].median())
        self.global_tenure_med_: float = float(df["YearsAtCompany"].median())

        # Store defaults for graceful legacy 18-feature fallback
        self.default_values_: Dict[str, Any] = {}
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                self.default_values_[col] = float(df[col].median())
            else:
                mode_vals = df[col].mode(dropna=True)
                self.default_values_[col] = mode_vals.iloc[0] if len(mode_vals) > 0 else "Unknown"

        sample_out = self._engineer(df)
        self.output_columns_: List[str] = list(sample_out.columns)
        self.categorical_columns_: List[str] = (
            sample_out.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        )
        self.numeric_columns_: List[str] = [
            c for c in self.output_columns_ if c not in self.categorical_columns_
        ]
        return self

    def transform(self, X: Any) -> pd.DataFrame:
        df = self._ensure_dataframe(X, fitting=False)
        out = self._engineer(df)
        return out[self.output_columns_]

    def _ensure_dataframe(self, X: Any, fitting: bool) -> pd.DataFrame:
        if isinstance(X, pd.DataFrame):
            # Detect legacy 18-column DataFrame
            if set(LEGACY_18_FEATURES).issubset(set(X.columns)) and "OverTime" not in X.columns:
                return self._expand_legacy_18(X[LEGACY_18_FEATURES].to_numpy())
            return X.copy()

        arr = np.asarray(X)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        if arr.shape[1] == len(LEGACY_18_FEATURES) and not fitting:
            return self._expand_legacy_18(arr)
        raise ValueError(
            f"Expected a pandas DataFrame with raw HR columns or a {len(LEGACY_18_FEATURES)}-column "
            f"legacy numpy array, got shape {arr.shape}."
        )

    def _expand_legacy_18(self, arr: np.ndarray) -> pd.DataFrame:
        n_rows = arr.shape[0]
        base = pd.DataFrame(arr, columns=LEGACY_18_FEATURES)
        full = pd.DataFrame(index=range(n_rows))

        # Populate all known columns with training defaults first
        for col, val in self.default_values_.items():
            full[col] = val

        # Override with the 16 direct numeric columns from legacy input
        for col in LEGACY_18_FEATURES[:16]:
            full[col] = pd.to_numeric(base[col], errors="coerce").fillna(
                self.default_values_.get(col, 0.0)
            )

        # Decode binary flags back to raw categorical values
        single_flags = pd.to_numeric(base["MaritalStatus_Single"], errors="coerce").fillna(0).astype(int)
        overtime_flags = pd.to_numeric(base["OverTime_Yes"], errors="coerce").fillna(0).astype(int)
        full["MaritalStatus"] = np.where(single_flags == 1, "Single", "Married")
        full["OverTime"] = np.where(overtime_flags == 1, "Yes", "No")
        return full

    def _engineer(self, df: pd.DataFrame) -> pd.DataFrame:
        d = df.copy()

        drop_cols = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]
        if self.drop_rate_columns:
            drop_cols.extend(["DailyRate", "HourlyRate", "MonthlyRate"])
        d = d.drop(columns=[c for c in drop_cols if c in d.columns])

        # Ordinal encoding of BusinessTravel (while keeping OHE categorical column too)
        d["TravelScore"] = d["BusinessTravel"].map(TRAVEL_SCORE_MAP).fillna(1).astype(float)
        ot = (d["OverTime"] == "Yes").astype(float)
        single = (d["MaritalStatus"] == "Single").astype(float)

        # Log-stabilized transforms for right-skewed numeric distributions
        d["LogMonthlyIncome"] = np.log1p(np.maximum(d["MonthlyIncome"].astype(float), 0.0))
        d["LogDistanceFromHome"] = np.log1p(np.maximum(d["DistanceFromHome"].astype(float), 0.0))
        d["LogYearsAtCompany"] = np.log1p(np.maximum(d["YearsAtCompany"].astype(float), 0.0))
        d["LogTotalWorkingYears"] = np.log1p(np.maximum(d["TotalWorkingYears"].astype(float), 0.0))
        d["LogYearsSinceLastPromotion"] = np.log1p(
            np.maximum(d["YearsSinceLastPromotion"].astype(float), 0.0)
        )

        # Composite Satisfaction & Engagement indices
        sat_cols = [
            "EnvironmentSatisfaction",
            "JobSatisfaction",
            "RelationshipSatisfaction",
            "WorkLifeBalance",
        ]
        d["TotalSatisfaction"] = d[sat_cols].sum(axis=1).astype(float)
        d["EngagementScore"] = d["TotalSatisfaction"] + d["JobInvolvement"].astype(float)
        d["MinSatisfaction"] = d[sat_cols + ["JobInvolvement"]].min(axis=1).astype(float)
        d["LowSatisfactionCount"] = (
            (d["EnvironmentSatisfaction"] == 1).astype(float)
            + (d["JobSatisfaction"] == 1).astype(float)
            + (d["RelationshipSatisfaction"] == 1).astype(float)
            + (d["WorkLifeBalance"] == 1).astype(float)
            + (d["JobInvolvement"] == 1).astype(float)
        )

        # Career trajectory, velocity & stagnation ratios (mitigates multicollinearity)
        twy = d["TotalWorkingYears"].astype(float)
        yac = d["YearsAtCompany"].astype(float)
        yir = d["YearsInCurrentRole"].astype(float)
        ysp = d["YearsSinceLastPromotion"].astype(float)
        ywm = d["YearsWithCurrManager"].astype(float)
        ncw = d["NumCompaniesWorked"].astype(float)
        inc = d["MonthlyIncome"].astype(float)
        age = d["Age"].astype(float)
        lvl = np.maximum(d["JobLevel"].astype(float), 1.0)

        d["IncomePerYearExp"] = inc / (twy + 1.0)
        d["IncomePerAge"] = inc / np.maximum(age, 1.0)
        d["IncomePerJobLevel"] = inc / lvl
        d["TenureRatio"] = yac / (twy + 1.0)
        d["PromotionStagnation"] = ysp / (yac + 1.0)
        d["RoleStagnation"] = yir / (yac + 1.0)
        d["ManagerLoyaltyRatio"] = ywm / (yac + 1.0)
        d["AvgYearsPerCompany"] = twy / (ncw + 1.0)
        d["CareerStartAge"] = age - twy
        d["JobHopperRate"] = ncw / (twy + 1.0)
        d["YearsWithoutPromoGap"] = yac - ysp

        # Leak-free group-relative pay & tenure indices
        role_med = d["JobRole"].map(self.role_income_med_).fillna(self.global_income_med_)
        lvl_med = d["JobLevel"].map(self.level_income_med_).fillna(self.global_income_med_)
        dept_med = d["Department"].map(self.dept_income_med_).fillna(self.global_income_med_)
        role_ten = d["JobRole"].map(self.role_tenure_med_).fillna(self.global_tenure_med_)

        d["IncomeVsRoleMedian"] = inc / (role_med + 1.0)
        d["IncomeVsLevelMedian"] = inc / (lvl_med + 1.0)
        d["IncomeVsDeptMedian"] = inc / (dept_med + 1.0)
        d["TenureVsRoleMedian"] = yac / (role_ten + 1.0)

        # High-signal EDA interaction & non-linear features
        no_stock = (d["StockOptionLevel"] == 0).astype(float)
        is_new_hire = (yac <= 1.0).astype(float)
        d["NoStockOptions"] = no_stock
        d["StockOptionCat"] = d["StockOptionLevel"].astype(int).astype(str)
        d["IsNewHire"] = is_new_hire
        d["OverTime_x_Single"] = ot * single
        d["OverTime_x_NoStock"] = ot * no_stock
        d["OverTime_x_FreqTravel"] = ot * (d["TravelScore"] == 2.0).astype(float)
        d["OverTime_x_LowSat"] = ot * (d["LowSatisfactionCount"] >= 1.0).astype(float)
        d["OverTime_x_LowIncome"] = ot * (inc < 4000.0).astype(float)
        d["OverTime_x_Distance"] = ot * d["DistanceFromHome"].astype(float)
        d["OverTime_x_NewHire"] = ot * is_new_hire
        d["Single_x_NoStock"] = single * no_stock
        d["BurnoutIndex"] = (
            ot * 2.0
            + d["TravelScore"] * 0.75
            + (4.0 - d["WorkLifeBalance"].astype(float)) * 0.75
            + (4.0 - d["JobSatisfaction"].astype(float)) * 0.5
        )
        return d

## 3. Preprocessing Pipeline Factory (`build_preprocessor`)

In [ ]:
def build_preprocessor(X_sample: pd.DataFrame, drop_rate_columns: bool = True) -> Pipeline:
    """
    Construct a leak-free preprocessing Pipeline:
      HRFeatureEngineer -> ColumnTransformer(PowerTransformer(Yeo-Johnson) + OneHotEncoder).

    Note: `X_sample` is only inspected to discover categorical vs numeric column names;
    actual fitting occurs strictly inside CV folds or on `X_train` only.
    """
    fe_probe = HRFeatureEngineer(drop_rate_columns=drop_rate_columns)
    fe_probe.fit(X_sample)
    num_cols = fe_probe.numeric_columns_
    cat_cols = fe_probe.categorical_columns_

    col_transformer = ColumnTransformer(
        transformers=[
            ("num", PowerTransformer(method="yeo-johnson", standardize=True), num_cols),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )

    return Pipeline(
        steps=[
            ("feature_engineer", HRFeatureEngineer(drop_rate_columns=drop_rate_columns)),
            ("column_transformer", col_transformer),
        ]
    )

## 4. CV-Optimized Soft-Voting Ensemble (`WeightedSoftVotingClassifier`)

In [ ]:
class WeightedSoftVotingClassifier(BaseEstimator, ClassifierMixin):
    """
    CV-Optimized Soft-Voting Ensemble Classifier.

    Learns non-negative simplex weights across diverse base estimators by minimizing
    regularized out-of-fold log-loss (preventing overfitting to any single model while
    automatically rewarding calibrated, complementary models).
    """

    def __init__(
        self,
        estimators: Sequence[Tuple[str, BaseEstimator]],
        cv: int = 5,
        l2_diversity_penalty: float = 0.015,
        random_state: int = 42,
    ) -> None:
        self.estimators = estimators
        self.cv = cv
        self.l2_diversity_penalty = l2_diversity_penalty
        self.random_state = random_state

    def fit(self, X: Any, y: Any) -> "WeightedSoftVotingClassifier":
        y_arr = np.asarray(y).astype(int)
        self.classes_ = np.unique(y_arr)
        n_models = len(self.estimators)

        # Generate leak-free out-of-fold probabilities to learn optimal blend weights
        skf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)
        oof_preds = np.zeros((len(y_arr), n_models), dtype=float)

        for fold_tr, fold_val in skf.split(X, y_arr):
            X_f_tr = X.iloc[fold_tr] if hasattr(X, "iloc") else X[fold_tr]
            X_f_val = X.iloc[fold_val] if hasattr(X, "iloc") else X[fold_val]
            y_f_tr = y_arr[fold_tr]

            for m_idx, (_, est) in enumerate(self.estimators):
                cloned = clone(est)
                cloned.fit(X_f_tr, y_f_tr)
                oof_preds[fold_val, m_idx] = cloned.predict_proba(X_f_val)[:, 1]

        # Simplex-constrained SLSQP optimization of log-loss + mild entropy/L2 diversity tie-breaker
        uniform = np.ones(n_models, dtype=float) / n_models

        def objective(w: np.ndarray) -> float:
            w_norm = w / np.sum(w)
            blend = np.clip(oof_preds @ w_norm, 1e-6, 1.0 - 1e-6)
            loss = log_loss(y_arr, blend)
            # Mild regularization toward balanced diversity so complementary trees aren't zeroed out
            reg = self.l2_diversity_penalty * float(np.sum((w_norm - uniform) ** 2))
            return float(loss + reg)

        result = minimize(
            objective,
            x0=uniform,
            method="SLSQP",
            bounds=[(0.02, 1.0)] * n_models,
            constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        )
        raw_w = np.clip(result.x, 0.0, 1.0)
        self.weights_ = raw_w / np.sum(raw_w)
        self.weight_map_: Dict[str, float] = {
            name: float(w) for (name, _), w in zip(self.estimators, self.weights_)
        }
        self.oof_probabilities_ = oof_preds

        # Fit final base estimators on the complete training data
        self.fitted_estimators_: List[Tuple[str, BaseEstimator]] = []
        for name, est in self.estimators:
            fitted = clone(est)
            fitted.fit(X, y_arr)
            self.fitted_estimators_.append((name, fitted))

        return self

    def predict_proba(self, X: Any) -> np.ndarray:
        probs = np.column_stack(
            [est.predict_proba(X)[:, 1] for _, est in self.fitted_estimators_]
        )
        pos_prob = np.clip(probs @ self.weights_, 0.0, 1.0)
        return np.column_stack([1.0 - pos_prob, pos_prob])

    def predict(self, X: Any) -> np.ndarray:
        pos_prob = self.predict_proba(X)[:, 1]
        return (pos_prob >= 0.5).astype(int)

## 5. Quick Interactive Verification (Optional Demo)

In [ ]:
from pathlib import Path

df_demo = pd.read_csv(Path('..') / 'data' / 'employee_attrition.csv')
X_demo = df_demo.drop(columns=['Attrition'])

pre_demo = build_preprocessor(X_demo, drop_rate_columns=True)
X_trans = pre_demo.fit_transform(X_demo)
feat_out = pre_demo.named_steps['column_transformer'].get_feature_names_out()
print(f'Raw shape        : {X_demo.shape}')
print(f'Transformed shape: {X_trans.shape} ({len(feat_out)} features)')

# Verify backward compatibility with legacy 18-feature app.py array
legacy_sample = np.array([[35, 10, 3, 3, 2, 3, 5000, 2, 12, 3, 1, 10, 5, 3, 2, 3, 0, 1]], dtype=float)
legacy_trans = pre_demo.transform(legacy_sample)
print(f'Legacy 18-col transformed shape: {legacy_trans.shape} (finite={np.all(np.isfinite(legacy_trans))})')